# Kinship Reasoning on CLUTRR: Direct LLM Answers vs Knowledge-Graph Paths


In [13]:
import os
import ast
import re
import logging
import pandas as pd
import networkx as nx
from openai import OpenAI

# =========================
# Setup
# =========================
logging.basicConfig(level=logging.INFO)

workspace_dir = "."
csv_path = f"{workspace_dir}/clutrr_sample_50.csv"
out_path = f"{workspace_dir}/clutrr_results.csv"

client = OpenAI(
    base_url=os.environ.get("LLM_BASE_URL", "http://localhost:1234/v1"),
    api_key="lm-studio"
)

MODEL_NAME = "qwen2.5-3b-instruct"

df = pd.read_csv(csv_path)
N_SAMPLES = 50
df = df.head(N_SAMPLES)

# =========================
# Labels
# =========================
LABELS = [
    "aunt","uncle","mother","father","grandmother","grandfather",
    "niece","nephew","daughter","son","sister","brother",
    "mother-in-law","father-in-law","daughter-in-law","son-in-law",
    "brother-in-law","sister-in-law","grandson","granddaughter","unknown"
]
LABELS_SET = set(LABELS)

# =========================
# Part 2 Constants & Regex
# =========================
ALLOWED_RELATIONS = {
    "has father", "has mother", "has brother", "has sister",
    "has son", "has daughter", "has husband", "has wife",
    "has grandmother", "has grandfather", "has grandson", "has granddaughter",
    "has uncle", "has aunt", "has nephew", "has niece",
    "has mother-in-law", "has father-in-law", "has son-in-law", "has daughter-in-law",
    "has brother-in-law", "has sister-in-law",
}

INLAW_RELATIONS = {
    "has mother-in-law", "has father-in-law", "has son-in-law", "has daughter-in-law",
    "has brother-in-law", "has sister-in-law",
}

TRIPLE_RE = re.compile(
    r'^\s*[-•]?\s*\(?\s*([^,()]+?)\s*,\s*([^,()]+?)\s*,\s*([^,()]+?)\s*\)?\s*\.?\s*$'
)

# =========================
# Utility
# =========================
def parse_query_tuple(q):
    a, b = ast.literal_eval(q)
    return str(a), str(b)

def clean_story_text(story: str) -> str:
    return str(story).replace("[", "").replace("]", "")

def norm_name(x: str) -> str:
    return re.sub(r"\s+", " ", (x or "").strip().lower())

def norm_rel(x: str) -> str:
    x = (x or "").strip().lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x

def norm_label(x: str) -> str:
    x = (x or "").strip().lower()
    x = re.sub(r"[^a-z\s-]", "", x)
    x = x.replace("-", " ")
    x = re.sub(r"\s+", " ", x).strip()
    x = x.replace("mother in law", "mother-in-law")
    x = x.replace("father in law", "father-in-law")
    x = x.replace("daughter in law", "daughter-in-law")
    x = x.replace("son in law", "son-in-law")
    x = x.replace("brother in law", "brother-in-law")
    x = x.replace("sister in law", "sister-in-law")
    return x

def extract_entities(story: str):
    ents = re.findall(r"\[([^\]]+)\]", str(story))
    return list(dict.fromkeys(ents))

def llm_generate(prompt: str, max_new_tokens: int = 600) -> str:
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "Output only what is requested. No extra text."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=max_new_tokens,
    )
    return (resp.choices[0].message.content or "").strip()

# =========================
# Part 1 prompt
# =========================
def prompt_direct_answer(story, p1, p2):
    labels_str = ", ".join(LABELS)
    return f"""You are answering a family-relation question using ONLY the facts in the story.

TASK:
Decide which SINGLE label from the POSSIBLE LABELS best describes
the relationship of PERSON_A with respect to PERSON_B.

PERSON_A = [{p2}]
PERSON_B = [{p1}]

POSSIBLE LABELS (choose exactly one):
{labels_str}

STORY:
{str(story)}

OUTPUT RULES (STRICT):
- Output EXACTLY ONE label (single line).
- Your answer MUST be exactly one of the POSSIBLE LABELS.
- Only use 'unknown' if there is genuinely not enough information.
"""

# =========================
# Part 4 prompt
# =========================
def prompt_path_answer(path_triples, p1, p2):
    labels_str = ", ".join(LABELS)

    if not path_triples:
        triples_block = "(no path found)"
    else:
        triples_block = "\n".join([f"({s}, {r}, {o})" for (s, r, o) in path_triples])

    return f"""You are answering a family-relation question using ONLY the given relationship triples.

TASK:
Choose ONE label from POSSIBLE LABELS that best describes:
Relationship of PERSON_A with respect to PERSON_B.

PERSON_A = [{p2}]
PERSON_B = [{p1}]

POSSIBLE LABELS (choose exactly one):
{labels_str}

EVIDENCE (ONLY these triples):
{triples_block}

ALLOWED REASONING (you may use these inference patterns ONLY if needed):
1) Direct mapping:
   - (X, has mother, Y) => Y is mother of X
   - (X, has father, Y) => Y is father of X
   - (X, has daughter, Y) => Y is daughter of X
   - (X, has son, Y) => Y is son of X
   - (X, has brother, Y) => Y is brother of X
   - (X, has sister, Y) => Y is sister of X

2) Grand relations:
   - If B has child C and C has mother A => A is mother of B's child (not necessarily in-law)
   - If A is mother of C and C is mother of D => A is grandmother of D
   - If A is mother of C and C is father of D => A is grandmother of D
   (similarly for grandfather with father)

3) Spouse-from-co-parents (common in CLUTRR):
   - If a child C has (C, has father, F) and (C, has mother, M),
     then infer F and M are spouses (unless contradicted).
   - This allows in-law relations:
     If (B, has spouse, S) and (S, has mother, A) => A is mother-in-law of B
     If (B, has spouse, S) and (S, has father, A) => A is father-in-law of B

IMPORTANT:
- Use ONLY the evidence triples and the allowed reasoning patterns above.
- If still ambiguous, output 'unknown'.

OUTPUT RULES (STRICT):
- Output EXACTLY ONE label (single line).
- Must be exactly one of the POSSIBLE LABELS.
"""

# =========================
# Part 2 Parsing
# =========================
def parse_llm_triples(text: str):
    triples = []
    for line in (text or "").splitlines():
        line = line.strip()
        if not line:
            continue
        m = TRIPLE_RE.match(line)
        if not m:
            continue
        s = norm_name(m.group(1))
        r = norm_rel(m.group(2))
        o = norm_name(m.group(3))
        if s and o and r in ALLOWED_RELATIONS:
            triples.append((s, r, o))

    seen, out = set(), []
    for t in triples:
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out

def parse_ground_truth(gt_string: str) -> set:
    gt = set()
    if not isinstance(gt_string, str) or not gt_string.strip():
        return gt

    pat = r'\(([^,]+),\s*(has\s+[^,]+),\s*([^)]+)\)'
    for s, r, o in re.findall(pat, gt_string):
        gt.add((norm_name(s), norm_rel(r), norm_name(o)))
    return gt

# =========================
# Gender inference
# =========================
def infer_gender_from_story(story_raw: str, entities):
    story = clean_story_text(story_raw).lower()
    genders = {norm_name(e): None for e in entities}

    for e in entities:
        # "Robert and his ..." / "Marie and her ..."
        if re.search(rf"\b{re.escape(e.lower())}\b\s+and\s+his\b", story):
            genders[norm_name(e)] = "male"
        elif re.search(rf"\b{re.escape(e.lower())}\b\s+and\s+her\b", story):
            genders[norm_name(e)] = "female"

    return genders

def filter_relations_by_story(triples, story_raw: str):
    story = clean_story_text(story_raw).lower()
    if "in-law" not in story:
        triples = [t for t in triples if t[1] not in INLAW_RELATIONS]
    return triples

def propagate_gender_from_triples(genders, triples):
    """
    FIXED:
    - Do NOT infer the SUBJECT gender from relations like 'has father'/'has mother'.
    - Infer OBJECT gender when relation implies gender of the object (father/mother/etc).
    - Infer spouse object gender from wife/husband.
    """
    def set_gender(name, g):
        if name and genders.get(name) is None:
            genders[name] = g

    for s, r, o in triples:
        if r == "has wife":
            set_gender(o, "female")
        elif r == "has husband":
            set_gender(o, "male")
        elif r == "has mother":
            set_gender(o, "female")
        elif r == "has father":
            set_gender(o, "male")
        elif r == "has son":
            set_gender(o, "male")
        elif r == "has daughter":
            set_gender(o, "female")
        elif r == "has grandmother":
            set_gender(o, "female")
        elif r == "has grandfather":
            set_gender(o, "male")
        elif r == "has grandson":
            set_gender(o, "male")
        elif r == "has granddaughter":
            set_gender(o, "female")
        elif r == "has brother":
            set_gender(o, "male")
        elif r == "has sister":
            set_gender(o, "female")
        elif r == "has uncle":
            set_gender(o, "male")
        elif r == "has aunt":
            set_gender(o, "female")
        # niece/nephew tell gender of the subject:
        elif r == "has niece":
            set_gender(s, "female")
        elif r == "has nephew":
            set_gender(s, "male")

    return genders

# =========================
# Spouse inference from co-parents
# =========================
def add_spouse_from_coparents(triples, genders):
    """
    If child C has both:
      (C, has father, F) and (C, has mother, M)
    infer:
      (F, has wife, M) and (M, has husband, F)
    Only if genders are consistent (F=male, M=female) to keep it safe.
    """
    tset = set(triples)

    mother_of = {}
    father_of = {}
    for c, r, p in tset:
        if r == "has mother":
            mother_of[c] = p
        elif r == "has father":
            father_of[c] = p

    added = set()
    for c in set(mother_of) & set(father_of):
        m = mother_of[c]
        f = father_of[c]
        if genders.get(f) == "male" and genders.get(m) == "female":
            added.add((f, "has wife", m))
            added.add((m, "has husband", f))

    return list(tset | added)

# =========================
# Reciprocal closure
# =========================
def add_safe_inverses(triples, genders):
    out = list(triples)

    def add(t):
        out.append(t)

    for s, r, o in triples:
        if r == "has wife":
            add((o, "has husband", s))
        elif r == "has husband":
            add((o, "has wife", s))
        elif r == "has brother":
            add((o, "has brother", s))
        elif r == "has sister":
            add((o, "has sister", s))
        elif r == "has mother":
            add((o, "has daughter", s))
        elif r == "has father":
            add((o, "has son", s))
        elif r == "has nephew":
            add((o, "has uncle", s))
        elif r == "has niece":
            add((o, "has aunt", s))

    out = [t for t in out if t[0] != t[2]]

    seen, dedup = set(), []
    for t in out:
        if t not in seen:
            seen.add(t)
            dedup.append(t)
    return dedup

# =========================
# Prompts (stage1/stage2)
# =========================
def prompt_stage1_extract_explicit(story_raw: str) -> str:
    story = clean_story_text(story_raw)
    allowed = ", ".join(sorted(ALLOWED_RELATIONS))
    entities = ", ".join(extract_entities(story_raw))

    return f"""Extract explicit family relationship triples from the story.

OUTPUT RULES (STRICT):
- Output ONLY triples, one per line.
- Format EXACTLY: (Name1, has relation, Name2)
- Use ONLY these relations:
  {allowed}
- Use ONLY names that appear in the story (from this list):
  {entities}
- Extract ONLY relationships explicitly stated in the text.
- DO NOT swap direction.

EXAMPLES:
Story: "Robert and his brother Antonio."
Output:
(Robert, has brother, Antonio)

Story: "Robert's daughter Michelle."
Output:
(Robert, has daughter, Michelle)

Story: "Robert asked his wife Marie."
Output:
(Robert, has wife, Marie)

Story: "Marie mentioned that her son Mario..."
Output:
(Marie, has son, Mario)

IMPORTANT:
- Do NOT invent in-law relations unless the story explicitly contains the phrase "in-law".

NOW EXTRACT FROM THIS STORY:
{story}

Output ONLY triples (one per line):
"""

def prompt_stage2_add_reciprocals(story_raw: str, stage1_triples_text: str) -> str:
    story = clean_story_text(story_raw)
    allowed = ", ".join(sorted(ALLOWED_RELATIONS))

    return f"""You will be given:
1) The original story
2) A list of explicit triples extracted from it

TASK:
- Add ONLY reciprocal triples for the given triples.
- Do NOT add any new non-reciprocal triples.
- Output ONLY triples, one per line.
- Use ONLY these relations:
  {allowed}

STRICT CONSTRAINTS:
- Do NOT invent in-law relations unless the story explicitly contains the phrase "in-law".
- Do NOT convert nephew/niece into parent/child (nephew/niece is NOT son/daughter).
- Do NOT generate nephew/niece from uncle/aunt. (Only the other direction is allowed.)
- Do NOT create uncle/aunt unless it is the reciprocal of a given nephew/niece triple.

Reciprocal rules:
- (X, has brother, Y)  => (Y, has brother, X)
- (X, has sister, Y)   => (Y, has sister, X)
- (X, has wife, Y)     => (Y, has husband, X)
- (X, has husband, Y)  => (Y, has wife, X)

Parent/child:
- (X, has daughter, Y) => (Y, has father, X) if X is male, or (Y, has mother, X) if X is female. If unclear, SKIP.
- (X, has son, Y)      => (Y, has father, X) if X is male, or (Y, has mother, X) if X is female. If unclear, SKIP.
- (X, has mother, Y)   => (Y, has daughter, X)
- (X, has father, Y)   => (Y, has son, X)

Niece/nephew (ONLY this direction):
- (X, has nephew, Y) => (Y, has uncle, X)
- (X, has niece, Y)  => (Y, has aunt, X)

IMPORTANT:
- If unclear, SKIP rather than guess.

STORY:
{story}

INPUT TRIPLES:
{stage1_triples_text}

Output ONLY triples (one per line):
"""

# =========================
# Consistency filters
# =========================
def apply_consistency_filters(triples, genders):
    tset = set(triples)

    supported_uncle_aunt = set()
    for s, r, o in tset:
        if r == "has nephew":
            supported_uncle_aunt.add((o, "has uncle", s))
        elif r == "has niece":
            supported_uncle_aunt.add((o, "has aunt", s))

    filtered = set()
    for s, r, o in tset:
        if r in {"has uncle", "has aunt"}:
            if (s, r, o) in supported_uncle_aunt:
                filtered.add((s, r, o))
            continue
        filtered.add((s, r, o))

    cleaned = set()
    for s, r, o in filtered:
        if r == "has father" and genders.get(o) == "female":
            continue
        if r == "has mother" and genders.get(o) == "male":
            continue
        cleaned.add((s, r, o))

    enforce_add = set()
    enforce_drop = set()
    for p, r, c in cleaned:
        if r in {"has son", "has daughter"}:
            pg = genders.get(p)
            if pg == "female":
                enforce_add.add((c, "has mother", p))
                enforce_drop.add((c, "has father", p))
            elif pg == "male":
                enforce_add.add((c, "has father", p))
                enforce_drop.add((c, "has mother", p))

    cleaned |= enforce_add
    cleaned -= enforce_drop

    return list(cleaned)

# =========================
# Scoring
# =========================
def compare_sets(pred_set: set, gt_set: set):
    tp = len(pred_set & gt_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return tp, fp, fn, precision, recall, f1

def format_triples_for_output(triples):
    if not triples:
        return "(no triples extracted)"
    lines = []
    for s, r, o in sorted(triples, key=lambda x: (x[0], x[1], x[2])):
        lines.append(f"({s}, {r}, {o})")
    return "\n".join(lines)

# =========================
# Shortest path
# =========================
def build_graph(triples, add_reverse=True):
    G = nx.MultiDiGraph()
    for s, p, o in triples:
        G.add_edge(s, o, relation=p, reversed=False)
        if add_reverse:
            G.add_edge(o, s, relation=f"{p}", reversed=True)
    return G

def _best_edge_data(edge_dict):
    candidates = list(edge_dict.values())
    candidates.sort(key=lambda d: (d.get("reversed", True), len(d.get("relation", ""))))
    return candidates[0] if candidates else {}

def find_shortest_path_triples(G, start, end):
    if start not in G or end not in G:
        return None
    try:
        nodes_path = nx.shortest_path(G, source=start, target=end)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None

    triples_path = []
    for u, v in zip(nodes_path, nodes_path[1:]):
        edge_dict = G.get_edge_data(u, v) or {}
        data = _best_edge_data(edge_dict)

        rel = data.get("relation", "UNKNOWN_RELATION")
        is_rev = data.get("reversed", False)

        if is_rev:
            triples_path.append((v, rel, u))  # emit original direction
        else:
            triples_path.append((u, rel, v))

    return triples_path

# =========================
# MAIN (Parts 1–4)
# =========================
df["p1"], df["p2"] = zip(*df["query"].map(parse_query_tuple))
has_target = "target_text" in df.columns
if has_target:
    df["target_norm"] = df["target_text"].map(norm_label)

pred_direct = []
direct_correct = []

pred_path = []
path_correct = []

extracted_triples_col = []
path_triples_col = []

kg_tp = []; kg_fp = []; kg_fn = []
kg_precision = []; kg_recall = []; kg_f1 = []

for i, row in df.iterrows():
    story_raw = row["story"]
    p1, p2 = row["p1"], row["p2"]

    # -------------------------
    # Part 1
    # -------------------------
    ans_direct = llm_generate(prompt_direct_answer(story_raw, p1, p2), max_new_tokens=60)
    pred_direct.append(ans_direct)
    if has_target:
        direct_correct.append(norm_label(ans_direct) == row["target_norm"])

    # -------------------------
    # Part 2 extraction
    # -------------------------
    gt = parse_ground_truth(row.get("ground_truth_", ""))

    s1_out = llm_generate(prompt_stage1_extract_explicit(story_raw), max_new_tokens=450)
    s1_triples = parse_llm_triples(s1_out)
    s1_triples = filter_relations_by_story(s1_triples, story_raw)

    entities = extract_entities(story_raw)
    genders = infer_gender_from_story(story_raw, entities)
    genders = propagate_gender_from_triples(genders, s1_triples)

    s2_prompt = prompt_stage2_add_reciprocals(story_raw, s1_out)
    s2_out = llm_generate(s2_prompt, max_new_tokens=650)
    s2_triples = parse_llm_triples(s2_out)
    s2_triples = filter_relations_by_story(s2_triples, story_raw)

    merged = list(dict.fromkeys(s1_triples + s2_triples))
    genders = propagate_gender_from_triples(genders, merged)

    merged = add_spouse_from_coparents(merged, genders)

    # inverses + consistency
    merged = add_safe_inverses(merged, genders)
    merged = apply_consistency_filters(merged, genders)

    pred = set(merged)

    # grandchild-only hallucination filter
    story_clean = clean_story_text(story_raw).lower()
    has_gp = any(w in story_clean for w in ["grandmother", "grandfather"])
    has_gc = any(w in story_clean for w in ["grandson", "granddaughter"])
    if has_gp and not has_gc:
        pred = {t for t in pred if t[1] not in {"has grandson", "has granddaughter"}}

    extracted_triples_col.append(sorted(list(pred)))

    tp, fp, fn, precision, recall, f1 = compare_sets(pred, gt)
    kg_tp.append(tp); kg_fp.append(fp); kg_fn.append(fn)
    kg_precision.append(precision); kg_recall.append(recall); kg_f1.append(f1)

    # -------------------------
    # Part 3: shortest path
    # -------------------------
    start = norm_name(p1)
    end = norm_name(p2)

    G = build_graph(pred, add_reverse=True)
    path_triples = find_shortest_path_triples(G, start, end)
    path_triples = path_triples if path_triples else []
    path_triples_col.append(path_triples)

    # -------------------------
    # Part 4: prompt
    # -------------------------
    ans_path = llm_generate(prompt_path_answer(path_triples, p1, p2), max_new_tokens=40).strip()
    ans_path_norm = norm_label(ans_path)
    if ans_path_norm not in LABELS_SET:
        ans_path_norm = "unknown"
    pred_path.append(ans_path_norm)

    if has_target:
        path_correct.append(ans_path_norm == row["target_norm"])

    # -------------------------
    # PRINT
    # -------------------------
    print("\n" + "=" * 80)
    print(f"SAMPLE {i}")
    print("=" * 80)
    print("Story:")
    print(story_raw)
    print()
    print("Query:", row["query"])
    if has_target:
        print("Target:", row["target_text"])
    print()
    print("Direct Answer (Part 1):", ans_direct)
    print("Path Answer   (Part 4):", ans_path_norm)
    print()
    print("EXTRACTED TRIPLES:")
    print(format_triples_for_output(pred))

    print("\n" + "-" * 60)
    print(f"Extracted: {len(pred)} | Expected: {len(gt)}")
    print(f"TP: {tp}, FP: {fp}, FN: {fn}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 Score:  {f1:.3f}")
    print("-" * 60)

    print("\nSHORTEST PATH (FACT-DIRECTION CORRECT):")
    if path_triples:
        for t in path_triples:
            print(f"  {t}")
    else:
        print("  No path found")
    print("=" * 80)

# =========================
# SUMMARY + SAVE
# =========================
df["pred_direct"] = pred_direct
df["pred_path"] = pred_path

if has_target:
    df["pred_direct_norm"] = df["pred_direct"].map(norm_label)
    df["pred_path_norm"] = df["pred_path"].map(norm_label)
    df["direct_correct"] = direct_correct
    df["path_correct"] = path_correct

df["extracted_triples"] = extracted_triples_col
df["path_triples"] = path_triples_col

df["kg_tp"] = kg_tp
df["kg_fp"] = kg_fp
df["kg_fn"] = kg_fn
df["kg_precision"] = kg_precision
df["kg_recall"] = kg_recall
df["kg_f1"] = kg_f1

print("\n" + "=" * 80)
print("SUMMARY (Parts 1–4)")
print("=" * 80)
print(f"Samples processed: {len(df)}")

if has_target and direct_correct:
    direct_acc = float(pd.Series(direct_correct).mean())
    print(f"Direct accuracy (Part 1): {direct_acc:.3f}")

if has_target and path_correct:
    path_acc = float(pd.Series(path_correct).mean())
    print(f"Path   accuracy (Part 4): {path_acc:.3f}")

avg_p = float(pd.Series(kg_precision).mean()) if kg_precision else 0.0
avg_r = float(pd.Series(kg_recall).mean()) if kg_recall else 0.0
avg_f = float(pd.Series(kg_f1).mean()) if kg_f1 else 0.0
print(f"Avg KG Precision (Part 2): {avg_p:.3f}")
print(f"Avg KG Recall    (Part 2): {avg_r:.3f}")
print(f"Avg KG F1        (Part 2): {avg_f:.3f}")

df.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")


INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



SAMPLE 0
Story:
[Robert] and his brother [Antonio] played harmonicas together. [Robert]'s daughter [Michelle] loved it. [Marie] and her daughter [Michelle] went to see a movie yesterday and then got ice cream afterwards. [Marie] just had a baby and presented the baby proudly to the new maternal grandmother, [Shantel].

Query: ('Robert', 'Shantel')
Target: mother-in-law

Direct Answer (Part 1): grandmother
Path Answer   (Part 4): mother-in-law

EXTRACTED TRIPLES:
(antonio, has brother, robert)
(marie, has daughter, michelle)
(marie, has mother, shantel)
(michelle, has father, robert)
(michelle, has mother, marie)
(robert, has brother, antonio)
(robert, has daughter, michelle)
(shantel, has daughter, marie)

------------------------------------------------------------
Extracted: 8 | Expected: 8
TP: 8, FP: 0, FN: 0
Precision: 1.000
Recall:    1.000
F1 Score:  1.000
------------------------------------------------------------

SHORTEST PATH (FACT-DIRECTION CORRECT):
  ('robert', 'has daug

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



SAMPLE 1
Story:
[Harold] was so excited to surprise his son, [Louis], with the tickets to the playoffs. [Louis] bought tickets for him to see the game with his nephew, [Mario]. [Robert] asked his wife [Marie] if she had any news about the family. [Marie] mentioned that her son, [Mario], got straight A's this semester at college.

Query: ('Robert', 'Harold')
Target: father-in-law

Direct Answer (Part 1): grandson
Path Answer   (Part 4): uncle

EXTRACTED TRIPLES:
(harold, has son, louis)
(louis, has father, harold)
(louis, has nephew, mario)
(marie, has husband, robert)
(marie, has son, mario)
(mario, has mother, marie)
(mario, has uncle, louis)
(robert, has wife, marie)

------------------------------------------------------------
Extracted: 8 | Expected: 8
TP: 8, FP: 0, FN: 0
Precision: 1.000
Recall:    1.000
F1 Score:  1.000
------------------------------------------------------------

SHORTEST PATH (FACT-DIRECTION CORRECT):
  ('robert', 'has wife', 'marie')
  ('marie', 'has son', 'm

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



SAMPLE 2
Story:
[Nicole]'s grandfather, [Harold], accompanied her to the basketball match [Beverly] went car shopping with her husband [Louis] and her daughter [Nicole]. [Harold] bought a new dress for his daughter [Marie].

Query: ('Louis', 'Marie')
Target: sister-in-law

Direct Answer (Part 1): aunt
Path Answer   (Part 4): aunt

EXTRACTED TRIPLES:
(beverly, has brother, louis)
(beverly, has husband, louis)
(beverly, has wife, louis)
(harold, has son, marie)
(louis, has brother, beverly)
(louis, has daughter, nicole)
(louis, has husband, beverly)
(louis, has wife, beverly)
(marie, has mother, harold)
(nicole, has grandmother, harold)
(nicole, has mother, louis)

------------------------------------------------------------
Extracted: 11 | Expected: 8
TP: 2, FP: 9, FN: 6
Precision: 0.182
Recall:    0.250
F1 Score:  0.211
------------------------------------------------------------

SHORTEST PATH (FACT-DIRECTION CORRECT):
  ('louis', 'has daughter', 'nicole')
  ('nicole', 'has grandmoth

... (90 outputs left out)

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



SAMPLE 48
Story:
[Constance] went to the mall, because she wanted to look for a present for her daughter, [Beatrice]. [Steven] was looking forward to dinner with his granddaughter, [Nadia]. [Dan] has a aunt named [Constance], routinely he went to school with his aunt [Don] and his brother [Sidney] had a tie score. [Sidney] asked his sister, [Beatrice], to come up with a tie-breaker. [Charles] went to the game with his sister [Nadia]. [Orville] got his son, [James], a car for his birthday. [Charles] was thrilled his brother, [Orville], was able to make it to the party. [Cesar] took his dad [Steven] to the game. [Cesar]'s son [Dan] was going to be playing

Query: ('Constance_2', 'Dan')
Target: nephew

Direct Answer (Part 1): aunt
Path Answer   (Part 4): unknown

EXTRACTED TRIPLES:
(beatrice, has brother, sidney)
(beatrice, has sister, sidney)
(cesar, has father, steven)
(constance, has brother, dan)
(dan, has brother, constance)
(sidney, has brother, beatrice)
(sidney, has sister, beatr

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



SAMPLE 49
Story:
[Nora] went to dinner with her daughter [Patrice]. They then went to the park afterwards. [Ellen] bought her mother, [Elsie], a puppy for her birthday. [James] is [Constance]'s older brother. He has never liked any of her boyfriends. [Nora] watched a golf tournament with her aunt [Ellen]. [Steve] and his son, [Stanley], went to the supermarket. [Sidney], [Steve]'s son, had asked them to bring him back some cookies. [Constance] and her son [Don] made hot dogs. [Don]'s brother [Sidney] ate one. [Margaret] was playing with her brother [James], but her mother [Elsie] told her she had to go to bed.

Query: ('Patrice', 'Stanley')
Target: uncle

Direct Answer (Part 1): niece
Path Answer   (Part 4): unknown

EXTRACTED TRIPLES:
(constance, has sister, james)
(constance, has son, don)
(don, has brother, sidney)
(don, has mother, constance)
(ellen, has mother, elsie)
(elsie, has daughter, ellen)
(james, has sister, constance)
(nora, has daughter, patrice)
(patrice, has mother, n